In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_dublin_core.xml", "data/metadata/climate_dataset_dublin_core.xml"),
    ("data/metadata/climate_dataset_datacite.xml", "data/metadata/climate_dataset_datacite.xml"),
    ("data/metadata/climate_dataset_schemaorg.jsonld", "data/metadata/climate_dataset_schemaorg.jsonld"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Bootstrap complete.


# Day 2: Comparing Dublin Core, DataCite, and schema.org

This notebook compares three metadata standards using the **same teaching dataset**.  
The goal is to help students see:

1. **what stays similar** across standards,
2. **what changes in structure and vocabulary**,
3. **which fields support FAIR most strongly**, and
4. **why different standards are useful in different contexts**.

We will work step by step:
- inspect the three files,
- extract key fields,
- build one comparison table,
- interpret the similarities and differences,
- connect the result to the FAIR principles.


## Step 1 — Import libraries and define file paths

We use:
- `xml.etree.ElementTree` for XML parsing,
- `json` for JSON-LD,
- `pandas` for clean comparison tables.

We keep everything local so the notebook runs reliably in class and in Colab.


In [2]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET
import pandas as pd

dc_path = Path("data/metadata/climate_dataset_dublin_core.xml")
datacite_path = Path("data/metadata/climate_dataset_datacite.xml")
schema_path = Path("data/metadata/climate_dataset_schemaorg.jsonld")

files_df = pd.DataFrame({
    "File": ["Dublin Core XML", "DataCite XML", "schema.org JSON-LD"],
    "Path": [str(dc_path), str(datacite_path), str(schema_path)],
    "Exists": [dc_path.exists(), datacite_path.exists(), schema_path.exists()]
})

files_df

,File,Path,Exists
0,Dublin Core XML,data/metadata/climate_dataset_dublin_core.xml,True
1,DataCite XML,data/metadata/climate_dataset_datacite.xml,True
2,schema.org JSON-LD,data/metadata/climate_dataset_schemaorg.jsonld,True


## Step 2 — Preview the three metadata files

Before parsing, it is useful to show students that the **same dataset** can be described in **different formats**:
- XML for Dublin Core,
- XML for DataCite,
- JSON-LD for schema.org.

This helps students separate:
- the **dataset itself**, from
- the **metadata representation format**.


In [3]:
print("=== Dublin Core (first 700 characters) ===")
print(dc_path.read_text(encoding="utf-8")[:700])

print("\n\n=== DataCite (first 700 characters) ===")
print(datacite_path.read_text(encoding="utf-8")[:700])

print("\n\n=== schema.org JSON-LD (first 700 characters) ===")
print(schema_path.read_text(encoding="utf-8")[:700])

=== Dublin Core (first 700 characters) ===
<?xml version="1.0" encoding="UTF-8"?>
<metadata
  xmlns:dc="http://purl.org/dc/elements/1.1/"
  xmlns:dcterms="http://purl.org/dc/terms/">

  <dc:title>2025 Global Climate Data</dc:title>

  <dc:creator>Global Climate Data Team</dc:creator>

  <dc:publisher>Open Research Repository (Teaching)</dc:publisher>

  <dc:identifier>10.1234/gcd-2025</dc:identifier>

  <dcterms:description>
    A small teaching dataset containing fictional annual climate summaries.
    Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.
  </dcterms:description>

  <dc:subject>climate</dc:subject>
  <dc:subject>temperature</dc:subjec


=== DataCite (first 700 characters) ===
<?xml version="1.0" encoding="UTF-8"?>
<resource
  xmlns="http://datacite.org/schema/kernel-4"
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
  xsi:schemaLocation="http://datacite.org/schema/kernel-4 http:

## Step 3 — Define lightweight parsing helpers

The parsers below are intentionally simple and classroom-friendly.

We extract a **common set of fields** so that students can compare standards side by side:
- title
- creator(s)
- identifier
- description
- publisher
- date or year
- format


In [4]:
def _local_name(tag: str) -> str:
    """Remove XML namespace to make tags easier to compare."""
    return tag.split('}', 1)[1] if '}' in tag else tag


def parse_dublin_core(xml_path: Path) -> dict:
    root = ET.parse(xml_path).getroot()

    def first(name):
        for el in root.iter():
            if _local_name(el.tag) == name:
                txt = (el.text or "").strip()
                if txt:
                    return txt
        return None

    creators = []
    for el in root.iter():
        if _local_name(el.tag) == "creator":
            txt = (el.text or "").strip()
            if txt:
                creators.append(txt)

    return {
        "standard": "Dublin Core",
        "title": first("title"),
        "creator": ", ".join(creators) if creators else None,
        "identifier": first("identifier"),
        "description": first("description"),
        "publisher": first("publisher"),
        "date_or_year": first("date"),
        "format": first("format"),
    }


def parse_datacite(xml_path: Path) -> dict:
    root = ET.parse(xml_path).getroot()

    def first(name):
        for el in root.iter():
            if _local_name(el.tag) == name:
                txt = (el.text or "").strip()
                if txt:
                    return txt
        return None

    creators = []
    for el in root.iter():
        if _local_name(el.tag) == "creatorName":
            txt = (el.text or "").strip()
            if txt:
                creators.append(txt)

    return {
        "standard": "DataCite",
        "title": first("title"),
        "creator": ", ".join(creators) if creators else None,
        "identifier": first("identifier"),
        "description": first("description"),
        "publisher": first("publisher"),
        "date_or_year": first("publicationYear"),
        "format": first("format"),
    }


def parse_schemaorg(json_path: Path) -> dict:
    data = json.loads(json_path.read_text(encoding="utf-8"))

    creators = []
    creator_field = data.get("creator")
    if isinstance(creator_field, list):
        for c in creator_field:
            if isinstance(c, dict):
                creators.append(c.get("name"))
            else:
                creators.append(str(c))
    elif isinstance(creator_field, dict):
        creators.append(creator_field.get("name"))
    elif creator_field:
        creators.append(str(creator_field))

    date_value = data.get("datePublished") or data.get("dateCreated") or data.get("dateModified")

    return {
        "standard": "schema.org",
        "title": data.get("name"),
        "creator": ", ".join([c for c in creators if c]) if creators else None,
        "identifier": data.get("identifier"),
        "description": data.get("description"),
        "publisher": data.get("publisher", {}).get("name") if isinstance(data.get("publisher"), dict) else data.get("publisher"),
        "date_or_year": date_value,
        "format": data.get("encodingFormat"),
    }

## Step 4 — Parse each file separately

We first inspect each standard on its own.  
This helps students understand the logic of each parser before seeing the merged comparison.


In [5]:
dc_meta = parse_dublin_core(dc_path)
datacite_meta = parse_datacite(datacite_path)
schema_meta = parse_schemaorg(schema_path)

print("Dublin Core:")
print(dc_meta)

print("\nDataCite:")
print(datacite_meta)

print("\nschema.org:")
print(schema_meta)

Dublin Core:
{'standard': 'Dublin Core', 'title': '2025 Global Climate Data', 'creator': 'Global Climate Data Team', 'identifier': '10.1234/gcd-2025', 'description': 'A small teaching dataset containing fictional annual climate summaries.\n    Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.', 'publisher': 'Open Research Repository (Teaching)', 'date_or_year': '2025-12-31', 'format': 'text/csv'}

DataCite:
{'standard': 'DataCite', 'title': '2025 Global Climate Data', 'creator': 'Global Climate Data Team', 'identifier': '10.1234/gcd-2025', 'description': 'A small teaching dataset containing fictional annual climate summaries.\n      Variables include average temperature, rainfall, and CO2 emissions for two fictional countries\n      (Alandia and Borealia) from 2021 to 2023.', 'publisher': 'Open Research Repository (Teaching)', 'date_or_year': '2025', 'format': None}

schema.org:
{'standard': 'schema.

## Step 5 — Build one shared comparison table

Now we map all three metadata records into one table.

This is the key teaching moment:  
even though the files use different syntax and field names, they still describe many of the **same core metadata ideas**.


In [ ]:
comparison_df = pd.DataFrame([dc_meta, datacite_meta, schema_meta])
comparison_df

## Step 6 — Compare field coverage more clearly

A useful classroom question is:

**Which standard gives us the field, and which standard leaves it missing or less explicit?**

The table below marks whether each key field is present.


In [6]:
coverage_df = comparison_df.copy()
for col in ["title", "creator", "identifier", "description", "publisher", "date_or_year", "format"]:
    coverage_df[col] = coverage_df[col].apply(lambda x: "Present" if pd.notna(x) and str(x).strip() else "Missing")

coverage_df

NameError: name 'comparison_df' is not defined

## Step 7 — Standard-by-standard interpretation

### Dublin Core
- simple and easy to understand,
- good for basic descriptive metadata,
- often used as an introductory standard.

### DataCite
- strong for citation, DOI-oriented dataset description,
- especially useful for repositories and scholarly data publication,
- usually stronger for formal dataset identification.

### schema.org
- strong for web discovery and machine-readable web metadata,
- useful for search engines and web indexing,
- common in public dataset discovery contexts.


## Step 8 — FAIR interpretation

Now connect the metadata comparison to FAIR.

### Findable
Strongly supported by:
- clear titles,
- identifiers,
- creators,
- publisher information.

### Accessible
Partly supported when metadata remains available and understandable even if the data file itself is not directly open.

### Interoperable
Supported by the use of recognized standards such as:
- Dublin Core,
- DataCite,
- schema.org.

### Reusable
Supported by:
- descriptive metadata,
- provenance-like information,
- publisher/date/format,
- structured documentation fields.


In [7]:
fair_view = pd.DataFrame({
    "FAIR Principle": ["Findable", "Accessible", "Interoperable", "Reusable"],
    "How metadata helps": [
        "Title, identifier, creator, and publisher make the dataset easier to discover.",
        "Structured metadata helps users understand how the dataset can be accessed and referenced.",
        "Using known standards makes metadata easier to exchange across tools and systems.",
        "Description, date, publisher, and format improve understanding and reuse."
    ],
    "Important fields in this example": [
        "title, identifier, creator",
        "identifier, description",
        "standardized field structures",
        "description, date_or_year, format, publisher"
    ]
})

fair_view

,FAIR Principle,How metadata helps,Important fields in this example
0,Findable,"Title, identifier, creator, and publisher make...","title, identifier, creator"
1,Accessible,Structured metadata helps users understand how...,"identifier, description"
2,Interoperable,Using known standards makes metadata easier to...,standardized field structures
3,Reusable,"Description, date, publisher, and format impro...","description, date_or_year, format, publisher"


## Step 9 — Weaknesses and limitations to discuss in class

Even when metadata exists, it may still be incomplete.

Possible discussion points:
- Is the **license** clearly visible in all standards?
- Is the **access information** explicit enough?
- Is the **format** consistently represented?
- Are **keywords** or **subject terms** strong enough for discovery?
- Does the metadata include enough context for real reuse?

These questions help students understand that **having metadata is not the same as having excellent metadata**.


In [8]:
issues_df = pd.DataFrame({
    "Potential issue": [
        "License not clearly represented in all three examples",
        "Format may be missing or inconsistent",
        "Access conditions may not be explicit",
        "Subject keywords may be limited",
        "Different field names make direct comparison harder"
    ],
    "Why it matters": [
        "A weak license description reduces reuse clarity.",
        "Missing format information reduces technical understanding.",
        "Poor access information weakens practical accessibility.",
        "Weak keywords reduce findability.",
        "Students must learn mapping between standards."
    ]
})

issues_df

,Potential issue,Why it matters
0,License not clearly represented in all three e...,A weak license description reduces reuse clarity.
1,Format may be missing or inconsistent,Missing format information reduces technical u...
2,Access conditions may not be explicit,Poor access information weakens practical acce...
3,Subject keywords may be limited,Weak keywords reduce findability.
4,Different field names make direct comparison h...,Students must learn mapping between standards.


## Step 10 — Final takeaway

Three important messages for students:

1. **Different metadata standards can describe the same dataset.**
2. **The field names and structures differ, but the underlying metadata goals are similar.**
3. **Better metadata directly supports FAIR, especially Findable and Reusable.**

This comparison is useful because it moves students from:
- memorizing standard names  
to
- understanding how metadata works in practice.


## Suggested class discussion questions

1. Which of the three standards looks easiest for a beginner to read?
2. Which standard seems strongest for scholarly dataset citation?
3. Which standard seems strongest for web discovery?
4. Which missing field would most reduce reusability?
5. Why is field mapping important when integrating metadata from multiple sources?
